# Step 18: Intermediate: Workflow Patterns with Typed Executors

        _Learner Notebook_  
        Level: **Intermediate**  
        Estimated time: **100 minutes**

        ## Learning Objectives
        - [ ] Use typed executors for clearer workflow contracts.
- [ ] Model fan-out, fan-in, and branch selection explicitly.
- [ ] Inspect workflow outputs and intermediate routing decisions.
- [ ] Practice workflow debugging with small deterministic nodes before adding live agents.

        ## Prerequisites
        - Core Steps 10 and 11 completed.


## Table of Contents

1. Setup & Imports
2. Part 1: Introduction
3. Part 2: Core Implementation
4. Part 3: Hands-On Exercises
5. Part 4: Solutions & Discussion
6. Part 5: Best Practices & Tips
7. Reflection & Next Experiments
8. Summary & Key Takeaways
9. Additional Resources


## Setup & Imports

Supplemental notebooks assume the core helpers are available and that you have already worked
through the matching core lessons. The import cell intentionally keeps the same bootstrap shape
as the core course so you can focus on the deeper pattern rather than environment setup.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from helpers import (
    LocalTfidfVectorStore,
    SQLiteConversationMemory,
    assert_minimum_python,
    chunk_documents,
    create_agent,
    create_chat_client,
    describe_response,
    load_settings,
    load_text_documents,
    print_banner,
    print_json,
    resolve_notebook_root,
    summarize_session,
    validate_provider_configuration,
)


## Part 1: Introduction

This notebook adds more depth to the workflow material by focusing on typed executors, graph shape, and the debugging mindset needed when data moves across several nodes.

This notebook is intentionally deeper than the core path. Expect more design discussion,
more implementation sections, and more open-ended exercises.


## Part 2: Core Implementation

### Create typed extractor and classifier nodes


In [ ]:
from typing_extensions import Never
from agent_framework import Executor, WorkflowBuilder, WorkflowContext, handler

class NormalizeRequest(Executor):
    @handler
    async def process(self, text: str, ctx: WorkflowContext[dict]) -> None:
        await ctx.send_message({"raw": text, "normalized": text.lower().strip()})

class ClassifyRequest(Executor):
    @handler
    async def process(self, payload: dict, ctx: WorkflowContext[dict]) -> None:
        normalized = payload["normalized"]
        route = "technical" if "python" in normalized or "workflow" in normalized else "general"
        await ctx.send_message({"route": route, **payload})


## Part 2: Core Implementation

### Add route-specific executors


In [ ]:
class TechnicalPath(Executor):
    @handler
    async def process(self, payload: dict, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(f"TECH ANALYSIS: {payload['raw']}")

class GeneralPath(Executor):
    @handler
    async def process(self, payload: dict, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(f"GENERAL ANALYSIS: {payload['raw']}")

class FinalReporter(Executor):
    @handler
    async def process(self, summary: str, ctx: WorkflowContext[Never, str]) -> None:
        await ctx.yield_output(f"FINAL OUTPUT: {summary}")

normalize = NormalizeRequest(id="normalize")
classify = ClassifyRequest(id="classify")
technical = TechnicalPath(id="technical")
general = GeneralPath(id="general")
reporter = FinalReporter(id="reporter")


## Part 2: Core Implementation

### Build branching and fan-in into one workflow


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=normalize)
    .add_edge(normalize, classify)
    .add_edge(classify, technical, condition=lambda payload: payload["route"] == "technical")
    .add_edge(classify, general, condition=lambda payload: payload["route"] == "general")
    .add_edge(technical, reporter)
    .add_edge(general, reporter)
    .build()
)

print((await workflow.run("How do workflow edges affect Python agent design?")).get_outputs())
print((await workflow.run("Give me a short summary of the course.")).get_outputs())


## Part 2: Core Implementation

### Inspect message-shape assumptions


In [ ]:
examples = [
    {"route": "technical", "raw": "Explain workflow fan-out."},
    {"route": "general", "raw": "Summarize the docs."},
]

print_json(
    {
        "normalize_output_shape": {"raw": "text", "normalized": "text"},
        "classify_output_examples": examples,
    },
    title="Workflow message contracts",
)


## Part 3: Hands-On Exercises

### Exercise 1
Add a third route for risk-sensitive requests, and create a dedicated executor for it.

### Exercise 2
Write a helper that prints which executor should receive a message based on the payload.

Work through both guided exercises before comparing against the solutions.


In [ ]:
# TODO: extend the routing logic with a risk-sensitive path


In [ ]:
def explain_route(payload: dict) -> str:
    # TODO: map payload route values to executor names
    return ""


## Part 4: Solutions & Discussion

Use this section to compare your implementation with one clear working approach. The goal is not
perfect matching output; the goal is a sound pattern that is easy to explain, debug, and extend.


In [ ]:
class RiskPath(Executor):
    @handler
    async def process(self, payload: dict, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(f"RISK REVIEW: {payload['raw']}")

risk = RiskPath(id="risk")
print("Created a dedicated risk-review executor.")


In [ ]:
def explain_route(payload: dict) -> str:
    mapping = {
        "technical": "technical",
        "general": "general",
        "risk": "risk",
    }
    return mapping.get(payload["route"], "unknown")

print_json(
    {
        "technical": explain_route({"route": "technical"}),
        "general": explain_route({"route": "general"}),
        "risk": explain_route({"route": "risk"}),
    },
    title="Exercise 2 solution",
)


## Part 5: Best Practices & Tips

        - Design message shapes before writing edge conditions.
- Use deterministic executors first when debugging graph behavior.
- Keep each executor narrowly responsible for one transformation step.
- Make route labels explicit so you can inspect them later.


## Reflection & Next Experiments

- Which part of `Intermediate: Workflow Patterns with Typed Executors` felt like the biggest jump from the core course?
- What would you keep deterministic before turning this into a live production feature?
- Where would you add tests, traces, or operator controls before shipping this pattern?


## Summary & Key Takeaways

You pushed past simple chains and treated workflows as typed graphs with inspectable contracts and debugging surfaces.


## Additional Resources

        - `core notebook: 10_basic_workflows.ipynb`
- `core notebook: 11_conditional_workflows.ipynb`
- `docs/references.md`
